# Section 4: Deep Neural Networks

**Student:** `<name>`  
**GitHub username:** `<username>`  
**Primary task:** `<regression / binary classification / multiclass classification>`

This notebook is the complete Section 4 assignment. Keep the headings, replace every placeholder, and add cells only when they help make the experiment reproducible. Read `../guide.md` and `../homework.md` before starting.

Use this evidence loop throughout: **predict what should happen → run one controlled check → interpret the result → name a limitation**.

## 0. Setup and reproducibility

Run the notebook on CPU unless you have a specific reason to use another device. Record the package versions and device. A seed reduces variation on one software/hardware setup but does not guarantee identical results everywhere.

In [ ]:
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
from sklearn.decomposition import PCA
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

SEED = 4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print({
    "python_device": str(DEVICE),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
})

## 1. Experiment contract

Complete this before training.

| Decision | Your choice | Why it is correct |
|---|---|---|
| Primary task and target | TODO | TODO |
| Train/validation/test row IDs | TODO | TODO |
| Section 3 baseline and saved result | TODO | TODO |
| MLP input columns | TODO | TODO |
| CNN sequence column | TODO | TODO |
| Model output shape | TODO | TODO |
| Target shape and dtype | TODO | TODO |
| Training loss | TODO | TODO |
| Probability/prediction conversion | TODO | TODO |
| Primary model-selection metric | TODO | TODO |

**Leakage risks known before training:** TODO

**Rule for opening the test set:** TODO

## 2. Manual forward pass and backpropagation

Use the exact network below. Show the calculations in Markdown or as clearly labeled intermediate values.

```text
x = [1.0, 2.0]                         y = 1

W1 = [[ 0.5, 0.25],                    b1 = [0.0, -0.5]
      [-0.5, 1.00]]

h = ReLU(W1 x + b1)

W2 = [0.8, -0.4]                       b2 = 0.1
logit = W2 h + b2
loss = binary cross-entropy with logits
```

Calculate the forward values, every parameter gradient, and one update with learning rate `0.1`. State why the loss should move in the observed direction.

In [ ]:
# TODO: calculate and print the manual forward pass, gradients, and updated loss.


### 2.1 Verify the trace with autograd

Recreate the same network with tensors. Compare the manual and autograd gradients parameter by parameter and report the maximum absolute difference.

In [ ]:
# TODO: reproduce the calculation with autograd and compare gradients.


### 2.2 Sigmoid and ReLU gradients

Plot both activations over a useful input range. Also plot or calculate their derivatives. Annotate saturation, the zero-gradient ReLU region, one advantage of ReLU, and one risk.

In [ ]:
# TODO: plot sigmoid, ReLU, and their derivatives.


## 3. Output–target–loss contract checks

Create three small untrained examples. For each, print output shape, target shape, target dtype, scalar loss, probabilities when relevant, and predictions. The loss must receive logits for classification.

In [ ]:
# TODO: regression contract check with MSELoss.


In [ ]:
# TODO: binary contract check with BCEWithLogitsLoss.


In [ ]:
# TODO: multiclass contract check with CrossEntropyLoss.


**Why are class indices usually preferable to one-hot targets for ordinary multiclass `CrossEntropyLoss`?** TODO

**How would the output, target, loss, and probability conversion change for multi-label classification?** TODO

**Why should sigmoid or softmax not appear before the classification losses above?** TODO

## 4. Load the fixed data split

Use repository-relative paths. Reuse the Section 3 target definition and row identifiers. If you must create a validation set, derive it from the old training portion and save the resulting identifiers.

In [ ]:
REPOSITORY_ROOT = Path.cwd()  # TODO: make robust if the notebook starts in another directory
DATA_PATH = REPOSITORY_ROOT / "TODO"

# TODO: load the data and reconstruct the fixed train/validation/test row IDs.


In [ ]:
# TODO: prove that row IDs are unique within each split and disjoint across splits.


## 5. Audit the exact model inputs

### 5.1 Tabular inputs for the MLP

Update the target distribution or class balance and the relevant Section 2 evidence: descriptive statistics, distributions, box plots, correlations, and PCA. Investigate missing, suspicious, target-derived, and leakage-prone values. Explain every feature exclusion.

In [ ]:
# TODO: audit the target and the exact MLP feature columns.


In [ ]:
# TODO: fit numerical preprocessing on training data only and transform all splits.
# Verify output shapes and at least one scaling property numerically.


### 5.2 Sequence inputs for the CNN

Report sequence lengths and character frequencies by split. Build a training-only vocabulary with explicit padding and unknown tokens. Choose and justify a fixed sequence length without consulting test outcomes. Report vocabulary size, unknown-token counts, padding rate, and truncation rate.

In [ ]:
# TODO: analyze sequence lengths and character frequencies.


In [ ]:
# TODO: build the training-only vocabulary and encode/pad/truncate all splits.
# Convert the sequence batch to the (N, C, L) shape expected by Conv1d.


**Why does the sequence representation have meaningful local order?** TODO

**Why would an arbitrary ordering of tabular columns not justify the same convolutional assumption?** TODO

**What information can character tokenization, padding, or truncation distort?** TODO

## 6. Shared training and evaluation utilities

Write small reusable functions for one training epoch, one validation epoch, checkpoint selection, prediction, metrics, and parameter count. Validation must not update parameters. Keep test evaluation unavailable until Section 9.

In [ ]:
# TODO: define DataLoaders and shared training/validation helpers.


## 7. MLP experiment

Define a small MLP that returns the unrestricted output required by your experiment contract. Show its architecture and parameter count. Print the input and output shapes for one batch.

In [ ]:
# TODO: define the MLP and verify its tensor contract on one batch.


### 7.1 Controlled comparison plan

| Run | Single main change | Prediction before training | Optimizer | Learning rate | Batch size | Regularization | Max epochs / stopping |
|---|---|---|---|---:|---:|---|---|
| MLP-A | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| MLP-B | TODO | TODO | TODO | TODO | TODO | TODO | TODO |

In [ ]:
# TODO: train both MLP runs; save histories and best validation checkpoints.


In [ ]:
# TODO: plot comparable training/validation learning curves.


**Selected MLP and validation rule:** TODO

**Did the controlled change behave as predicted?** TODO

**Evidence of underfitting, overfitting, unstable optimization, or no clear problem:** TODO

**Alternative explanation or limitation:** TODO

## 8. 1D CNN experiment

Define a small CNN with at least one `Conv1d`, a nonlinear activation, pooling, and a final task output. Print the tensor shape after every major stage for one batch.

In [ ]:
# TODO: define the CNN, report parameter count, and verify all major shapes.


### 8.1 Controlled comparison plan

| Run | Single main change | Prediction before training | Optimizer | Learning rate | Batch size | Regularization | Max epochs / stopping |
|---|---|---|---|---:|---:|---|---|
| CNN-A | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| CNN-B | TODO | TODO | TODO | TODO | TODO | TODO | TODO |

In [ ]:
# TODO: train both CNN runs; save histories and best validation checkpoints.


In [ ]:
# TODO: plot comparable training/validation learning curves.


**Selected CNN and validation rule:** TODO

**Did the controlled change behave as predicted?** TODO

**What can one convolutional filter respond to in this input?** TODO

**Evidence of underfitting, overfitting, unstable optimization, or no clear problem:** TODO

**Alternative explanation or limitation:** TODO

## 9. Lock choices, evaluate the test set once, and compare

Before running the next cell, write the selected checkpoints, any probability threshold, the evaluation metrics, and the reason each choice was made from training/validation evidence.

**Locked evaluation plan:** TODO

In [ ]:
# TODO: evaluate the locked MLP and CNN checkpoints on the untouched test set.
# Produce all task-required metrics and diagnostic plots.


In [ ]:
# TODO: make one comparison table with the Section 3 baseline, MLP, and CNN.
# Include representation, parameter count, best validation result, test metrics,
# approximate training time, and random seed.


### Comparison conclusions

1. **Did either neural network outperform the baseline by a practically meaningful amount?** TODO

2. **What evidence of overfitting or underfitting appears in the curves?** TODO

3. **Because the MLP and CNN use different inputs, what can and cannot be concluded from their score difference?** TODO

4. **What are the two most important threats to validity?** TODO

5. **Which model would you carry into Section 5, and why?** TODO

## 10. Critical review and reproducibility check

Restart the kernel and run all cells. Record the result. Then ask Codex to review tensor shapes, double activation, leakage, test-set tuning, evaluation mode, and unsupported claims.

**Restart-and-run-all result:** TODO

**One Codex suggestion accepted:** TODO

**Evidence for accepting it:** TODO

**One Codex suggestion changed or rejected:** TODO

**Evidence for changing or rejecting it:** TODO

**Final limitation that remains after review:** TODO